# 4. Input Adapters

Use this notebook when your workflow starts from an external coordinate,
biomolecular, polymer, spatial-network, or surface-mesh file. The public input
adapters separate source-specific parsing from the common geometric objects used
by the rest of `KnottedGraph`.

By the end, you will know how to:

- choose an adapter without guessing from a filename;
- inspect the returned coordinates, graph or mesh, metadata, and validation issues;
- make closure and unit choices explicitly; and
- continue from a normalized object to visualization, extraction, projection, or invariant calculation.

![Overview of input families normalized by KnottedGraph](../doc/assets/site_figures/input_formats_overview.png)

> The format labels in this overview are representative entry points. Some are
> public file readers, while fields, volumes, Hamiltonians, and several network
> serializations still require application-level conversion. The table below is
> the authoritative support summary for the public adapter layer.

## 4.1 Choose the input route

| Starting data | Public entry point | Normalized output | Important scope |
| --- | --- | --- | --- |
| In-memory `(N, 3)` coordinates; CSV, DAT, JSON, NPY, TSV, TXT, XYZ | `from_coordinate_chain` | `CoordinateInputResult.graph` | One ordered curve; JSON contains `points` or `coords`, and NPY is an `(N, 3)` array |
| PDB protein or nucleic-acid records | `from_protein_ca_backbone`, `from_nucleic_acid_backbone`, `from_pdb_backbone` | `PDBBackboneInputResult.graph` | One selected atom trace from one model and chain |
| mmCIF atom-site records | `from_mmcif_backbone` | `MMCIFBackboneInputResult.graph` | One selected atom trace from one model and chain |
| GROMACS GRO snapshot | `from_gromacs_gro` | `PolymerInputResult.graph` | One ordered, optionally filtered chain; coordinates are scaled on input |
| LAMMPS dump | `from_lammps_dump` | `PolymerInputResult.graph` | First frame, unscaled `x/y/z`, optionally one molecule ID |
| Paired node and edge CSV tables | `from_spatial_graph_csv` | `SpatialGraphInputResult.graph` | Connectivity plus node positions; optional curved `points_json` per edge |
| OBJ, OFF, PLY, STL, VTK, VTP surface mesh | `from_surface_mesh` | `SurfaceInputResult.mesh` | Requires the `surface` extra; returns `pyvista.PolyData`, not a skeleton graph |
| Hamiltonian, scalar volume, vector field, NPZ field archive | application workflow | workflow-specific surface, streamlines, or graph | Not a generic public file adapter |
| SWC, GraphML, standalone spatial-graph JSON, abstract edge list | user/application conversion to the common graph contract | `networkx.MultiGraph` | Not currently parsed by `knotted_graph.inputs` |

Install only the optional mesh stack when it is needed:

```bash
pip install "knotted_graph[surface]"
```

The examples below create all source files in a temporary directory and never
download data.

## Set up the offline examples

When working from a source checkout, this cell locates `src/` in the same way as
the other User Guide notebooks. An installed package needs no path adjustment.

In [ ]:
from pathlib import Path
import csv
import importlib.util
import json
import sys
import tempfile

import networkx as nx
import numpy as np

SEARCH_ROOT = Path.cwd().resolve()
while not (SEARCH_ROOT / "src").exists() and SEARCH_ROOT != SEARCH_ROOT.parent:
    SEARCH_ROOT = SEARCH_ROOT.parent

if (SEARCH_ROOT / "src").exists():
    PROJECT_ROOT = SEARCH_ROOT
    SRC_ROOT = PROJECT_ROOT / "src"
    if str(SRC_ROOT) not in sys.path:
        sys.path.insert(0, str(SRC_ROOT))
    PACKAGE_MODE = f"source checkout: {PROJECT_ROOT}"
elif importlib.util.find_spec("knotted_graph") is not None:
    PROJECT_ROOT = None
    PACKAGE_MODE = "installed package"
else:
    raise RuntimeError(
        "Could not locate a repository src directory or an installed "
        "knotted_graph package."
    )

from knotted_graph.core import validate_embedding
from knotted_graph.inputs import (
    from_coordinate_chain,
    from_gromacs_gro,
    from_lammps_dump,
    from_mmcif_backbone,
    from_protein_ca_backbone,
    from_spatial_graph_csv,
    write_gro_coords,
    write_lammps_dump,
)

TEMPORARY_EXAMPLES = tempfile.TemporaryDirectory(prefix="knottedgraph-inputs-")
EXAMPLE_DIR = Path(TEMPORARY_EXAMPLES.name)
print(f"package mode = {PACKAGE_MODE}")
print(f"temporary example directory = {EXAMPLE_DIR}")
print(f"surface extra available = {importlib.util.find_spec('pyvista') is not None}")

## 4.2 The common graph contract

Graph-valued adapters return an undirected `networkx.MultiGraph` with:

- a finite three-dimensional `pos` array on every node;
- a finite `(N, 3)` `pts` polyline on every edge; and
- polyline endpoints that agree with the positions of the incident nodes.

The result object keeps the parsed coordinates and source-specific fields alongside
the graph. `result.issues` is a list of non-fatal validation observations. Fatal
schema failures, missing required columns, ambiguous PDB chains, and insufficient
usable geometry raise exceptions instead of silently inventing data. Recoverable
PDB/mmCIF coordinate-row problems may be skipped and reported in `result.issues`.

In [ ]:
def report_graph_result(label, result):
    graph = result.graph
    contract_issues = validate_embedding(graph)
    print(f"{label}: {type(result).__name__}")
    print(f"  nodes / edges = {graph.number_of_nodes()} / {graph.number_of_edges()}")
    print(f"  source format = {getattr(result, 'source_format', graph.graph.get('source_format'))}")
    print(f"  adapter issues = {result.issues}")
    print(f"  graph-contract issues = {contract_issues}")
    assert isinstance(graph, nx.MultiGraph)
    assert contract_issues == []


def first_edge_points(graph):
    return np.asarray(next(iter(graph.edges(data=True)))[2]["pts"], dtype=float)


print("graph inspection helpers ready")

## 4.3 Ordered coordinate chains

The coordinate adapter accepts an in-memory array or a supported lightweight file.
It represents an open curve by two endpoint nodes and one polyline edge. Closure is
never guessed from scientific context.

In [ ]:
coordinate_csv = EXAMPLE_DIR / "coordinate_chain.csv"
coordinate_csv.write_text(
    "x,y,z\n"
    "0.0,0.0,0.0\n"
    "0.8,0.2,0.3\n"
    "1.3,1.0,0.1\n"
    "2.0,1.2,-0.2\n"
)

coordinate_result = from_coordinate_chain(
    coordinate_csv,
    input_id="demo_coordinate_chain",
    metadata={"length_unit": "arbitrary", "purpose": "offline tutorial"},
)
report_graph_result("coordinate CSV", coordinate_result)
print("  coordinate shape =", coordinate_result.coords.shape)
print("  provenance =", {key: coordinate_result.graph.graph[key] for key in ["input_id", "source_path", "source_format"]})

### Make closure explicit

If the first and last samples differ, `closed=True` alone is rejected. Use
`closure="direct"` only when the straight segment from the last sample to the
first is part of the intended embedding. `closure="metadata_only"` records an
intended closure but deliberately leaves the graph open, so it is not a substitute
for geometric closure before projection.

In [ ]:
open_triangle = np.array(
    [[0.0, 0.0, 0.0], [1.0, 0.0, 0.2], [0.2, 1.0, -0.1]],
    dtype=float,
)

try:
    from_coordinate_chain(open_triangle, closed=True)
except ValueError as error:
    print("expected explicit-closure error:", error)

closed_result = from_coordinate_chain(
    open_triangle,
    input_id="directly_closed_triangle",
    closed=True,
    closure="direct",
)
closed_points = first_edge_points(closed_result.graph)
report_graph_result("directly closed coordinate chain", closed_result)
print("  stored edge points =", closed_points.shape[0])
print("  first equals last =", np.allclose(closed_points[0], closed_points[-1]))

## 4.4 Local PDB and mmCIF atom traces

The biomolecular readers select one atom name, model, and chain and preserve residue
records as provenance. Passing a four-character PDB ID can download from RCSB, but
the examples here use local files and therefore require no network access. PDB files
with multiple matching chains require an explicit `chain_id`. The current mmCIF
reader targets RCSB-style `_atom_site` loops with one complete atom row on each
physical line; reformat other valid CIF layouts before loading them.

These adapters produce **ordered backbone or atom traces**. Constructing a contact,
interaction, vascular, or other domain-specific biological network is a separate
modeling decision; it is not inferred by the file reader.

In [ ]:
def pdb_atom(serial, atom_name, residue_name, chain_id, residue_id, x, y, z):
    return (
        f"ATOM  {serial:5d} {atom_name:>4s} {residue_name:>3s} {chain_id:1s}"
        f"{residue_id:4d}    {x:8.3f}{y:8.3f}{z:8.3f}"
        "  1.00 20.00           C\n"
    )


pdb_path = EXAMPLE_DIR / "mini_protein.pdb"
pdb_path.write_text(
    "".join(
        [
            pdb_atom(1, "CA", "ALA", "A", 1, 0.0, 0.0, 0.0),
            pdb_atom(2, "CA", "GLY", "A", 2, 1.1, 0.2, 0.1),
            pdb_atom(3, "CA", "SER", "A", 3, 1.7, 1.0, -0.2),
            pdb_atom(4, "CA", "VAL", "A", 4, 2.5, 1.1, 0.4),
        ]
    )
)

pdb_result = from_protein_ca_backbone(
    pdb_path,
    pdb_id="DEMO",
    chain_id="A",
)
report_graph_result("local PDB C-alpha trace", pdb_result)
print("  selected chain / atom =", pdb_result.chain_id, pdb_result.atom_name)
print("  first residue record =", pdb_result.records[0])

In [ ]:
mmcif_path = EXAMPLE_DIR / "mini_rna.cif"
mmcif_path.write_text(
    "\n".join(
        [
            "data_DEMO",
            "loop_",
            "_atom_site.group_PDB",
            "_atom_site.auth_atom_id",
            "_atom_site.label_atom_id",
            "_atom_site.auth_asym_id",
            "_atom_site.label_asym_id",
            "_atom_site.pdbx_PDB_model_num",
            "_atom_site.label_alt_id",
            "_atom_site.Cartn_x",
            "_atom_site.Cartn_y",
            "_atom_site.Cartn_z",
            "_atom_site.auth_comp_id",
            "_atom_site.label_comp_id",
            "_atom_site.auth_seq_id",
            "_atom_site.label_seq_id",
            "ATOM P P A A 1 . 0.0 0.0 0.0 A A 1 1",
            "ATOM P P A A 1 . 0.8 0.2 0.1 C C 2 2",
            "ATOM P P A A 1 . 1.3 0.9 -0.1 G G 3 3",
            "ATOM P P A A 1 . 2.0 1.1 0.3 U U 4 4",
            "#",
        ]
    )
    + "\n"
)

mmcif_result = from_mmcif_backbone(
    mmcif_path,
    pdb_id="DEMO",
    chain_id="A",
    atom_name="P",
)
report_graph_result("local mmCIF phosphorus trace", mmcif_result)
print("  selected chain / atom =", mmcif_result.chain_id, mmcif_result.atom_name)
print("  source URL =", mmcif_result.source_url)

## 4.5 Polymer snapshots: GRO and LAMMPS

A GRO file normally stores positions in nanometers. `from_gromacs_gro` multiplies
them by `output_unit_scale=10.0` by default, producing angstrom-like coordinates.
Set the scale explicitly when your convention differs. The LAMMPS reader uses the
first frame, requires unscaled `x`, `y`, and `z` columns, and sorts atoms by `id`
unless another available column is requested. Neither reader guesses polymer
bonding beyond the stored coordinate order.

In [ ]:
polymer_coords = np.array(
    [
        [0.0, 0.0, 0.0],
        [1.0, 0.5, 0.0],
        [1.5, 1.0, 0.5],
        [1.0, 1.5, 1.0],
        [0.0, 1.0, 1.0],
    ],
    dtype=float,
)

gro_path = EXAMPLE_DIR / "demo_polymer.gro"
write_gro_coords(polymer_coords, gro_path, residue_name="POL", atom_name="BB")
gro_result = from_gromacs_gro(
    gro_path,
    residue_name="POL",
    atom_name="BB",
    output_unit_scale=10.0,
    polymer_id="gro_demo",
)
report_graph_result("GROMACS GRO", gro_result)
np.testing.assert_allclose(gro_result.coords, polymer_coords)

lammps_path = EXAMPLE_DIR / "demo_polymer.dump"
write_lammps_dump(polymer_coords, lammps_path, molecule_id=7)
lammps_result = from_lammps_dump(
    lammps_path,
    molecule_id=7,
    polymer_id="lammps_demo",
)
report_graph_result("LAMMPS dump", lammps_result)
np.testing.assert_allclose(lammps_result.coords, polymer_coords)

## 4.6 Embedded spatial graphs from paired CSV tables

This adapter uses one node table and one edge table. Nodes require an identifier and
`x`, `y`, `z`; edges require `source` and `target`. An edge can contain a JSON-encoded
polyline in `points_json`. Its first and last points must match the endpoint node
positions. Extra columns are retained as attributes, and parallel edge IDs remain
distinct MultiGraph keys.

In [ ]:
nodes_csv = EXAMPLE_DIR / "network_nodes.csv"
edges_csv = EXAMPLE_DIR / "network_edges.csv"

with nodes_csv.open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerows(
        [
            ["node_id", "x", "y", "z", "type"],
            ["u", 0.0, 0.0, 0.0, "junction"],
            ["v", 1.0, 0.0, 0.0, "junction"],
            ["w", 0.5, 1.0, 0.4, "terminal"],
        ]
    )

with edges_csv.open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerows(
        [
            ["edge_id", "source", "target", "points_json", "type"],
            ["upper", "u", "v", json.dumps([[0, 0, 0], [0.5, 0.35, 0.2], [1, 0, 0]]), "cable"],
            ["lower", "u", "v", json.dumps([[0, 0, 0], [0.5, -0.3, -0.1], [1, 0, 0]]), "cable"],
            ["branch", "v", "w", "", "branch"],
        ]
    )

spatial_result = from_spatial_graph_csv(
    nodes_csv,
    edges_csv,
    graph_id="paired_csv_demo",
    metadata={"domain": "tutorial network", "length_unit": "m"},
)
report_graph_result("paired spatial-graph CSV", spatial_result)
print("  edge keys =", sorted(key for _, _, key in spatial_result.graph.edges(keys=True)))
print("  upper polyline =", spatial_result.graph.edges["u", "v", "upper"]["pts"] )

## 4.7 Surface meshes

The mesh adapter loads, optionally cleans, and optionally triangulates a surface. It
reports open boundary edges but does not invent a graph spine. Skeletonization or
another surface-to-graph construction is a separate, topology-sensitive workflow.
The cell prints an installation hint instead of failing when PyVista is unavailable.

In [ ]:
off_path = EXAMPLE_DIR / "tetrahedron.off"
off_path.write_text(
    "\n".join(
        [
            "OFF",
            "4 4 0",
            "0 0 0",
            "1 0 0",
            "0 1 0",
            "0 0 1",
            "3 0 2 1",
            "3 0 1 3",
            "3 1 2 3",
            "3 2 0 3",
        ]
    )
    + "\n"
)

if importlib.util.find_spec("pyvista") is None:
    surface_result = None
    print('Surface example skipped. Install with: pip install "knotted_graph[surface]"')
else:
    from knotted_graph.inputs import from_surface_mesh

    surface_result = from_surface_mesh(off_path, mesh_id="tetrahedron")
    print("surface mesh:", type(surface_result).__name__)
    print("  points / cells =", surface_result.mesh.n_points, surface_result.mesh.n_cells)
    print("  source format =", surface_result.source_format)
    print("  mesh issues =", surface_result.issues)
    assert surface_result.issues == []

## 4.8 Validation, units, and provenance checklist

Before using an imported object in a scientific conclusion, check all of the
following:

1. **Identity and selection:** record the source path or database ID, model, chain,
   atom filter, molecule ID, and any row-order assumption.
2. **Units:** `KnottedGraph` has no global length unit. Coordinate tables and meshes
   retain their numeric scale; GRO applies `output_unit_scale`; LAMMPS uses the
   unscaled values stored in the selected frame.
3. **Closure:** inspect the actual edge polyline. A metadata statement that a curve
   should be closed does not add missing geometry.
4. **Connectivity:** an ordered coordinate file creates one curve; it does not encode
   a branched network. Use paired spatial-graph CSV or construct a MultiGraph when
   incidence matters.
5. **Validation:** inspect `result.issues` and run `validate_embedding(result.graph)`.
   An empty list means the structural contract passed, not that the scientific model
   or topology has been independently verified.

Common failures are deliberately actionable: missing coordinate columns identify the
required names; ambiguous PDB chains list available choices; non-finite coordinates
are rejected; curved CSV edges must meet their endpoint positions; and open meshes
report boundary edges.

In [ ]:
graph_results = [
    coordinate_result,
    closed_result,
    pdb_result,
    mmcif_result,
    gro_result,
    lammps_result,
    spatial_result,
]

for result in graph_results:
    assert result.issues == []
    assert validate_embedding(result.graph) == []

print(f"validated {len(graph_results)} graph-valued adapter results")
if surface_result is not None:
    print("validated 1 mesh-valued adapter result")

## 4.9 Continue from the normalized object

For a graph-valued result, the common handoff is always `result.graph`:

The Plotly visualization in this optional snippet requires
`pip install "knotted_graph[viz]"`.

```python
import sympy as sp

graph = spatial_result.graph
Y = sp.Symbol("Y")

from knotted_graph.visualization import plot_3D_graph_plotly
figure = plot_3D_graph_plotly(graph)
figure.show()

from knotted_graph.projection import compute_yamada_polynomial
result = compute_yamada_polynomial(graph, Y, n_jobs=1, return_result=True)
print(result.polynomial)
print(result.projection.num_crossings)
```

Only send a graph to projection after inspecting its geometry, connectivity, and
closure. Surface meshes first need an application-appropriate graph extraction
step; fields and Hamiltonians likewise need streamline integration, skeletonization,
or another explicit reconstruction.

Continue with:

- **[02 — Core Workflows](02_core_workflows.ipynb)** for extraction, simplification,
  projection selection, and PD-code inspection;
- **[01 — Getting Started](01_getting_started.ipynb)** for the shortest complete
  graph-to-Yamada example; and
- the application notebooks for physical models or domain-specific mappings.

The publication galleries provide two complementary visual checks on these routes:
[distinct nonzero Yamada examples](../doc/assets/site_figures/input_yamada_nonzero.png)
show heterogeneous normalized graphs entering one invariant pipeline, while
[skeletonization beyond Yamada](../doc/assets/site_figures/input_skeletonization_beyond_yamada.png)
shows imported or reconstructed geometry used for morphology, flow, and structural
analysis without displaying a polynomial.